## 10.2 LSTM - 前向传播&矩阵计算

#### 1、先从单时间步开始

##### 1.1 学习顺序：单时间步 → 多时间步 → 加入 batch
LSTM 虽然看起来结构比普通 RNN 复杂很多，但它的本质仍然没有变：

整个序列的计算，本质上就是单时间步计算在时间维度上的重复。

所以我们学习 LSTM 前向传播时，最合理的顺序一定是：

- 先看一个时间步内部到底做了什么
- 再看这个过程如何沿着时间维重复
- 最后再看当一次输入多条样本时，维度如何扩展

这个思路和我们前面学习 RNN 是完全一致的。

##### 1.2 这一节的核心目标

- 一个 LSTM 单元在时刻 $t$ 到底接收什么输入？
- 它内部各个门按什么顺序计算？
- 细胞状态 $C_t$ 和隐藏状态 $h_t$ 是如何得到的？
- 整个序列为什么只是“重复单步”？
- 加入 batch 之后，张量维度如何变化？

#### 2、单时间步前向传播：先看输入和输出

##### 2.1 单时间步 LSTM 接收哪些输入
在某一个时间步 $t$，LSTM 接收三部分输入：

- 当前时刻输入：$x_t$
- 上一时刻隐藏状态：$h_{t-1}$
- 上一时刻细胞状态：$C_{t-1}$

所以，单时间步 LSTM 不是只看当前输入 $x_t$，而是还要结合“前面传下来的状态”。

可以写成：

$(x_t,\ h_{t-1},\ C_{t-1})$

##### 2.2 单时间步 LSTM 输出什么
经过当前时间步内部计算之后，会输出两个结果：

- 当前隐藏状态：$h_t$
- 当前细胞状态：$C_t$

写成：

$(h_t,\ C_t)$

其中：

- $h_t$ 会用于当前时刻的输出，或者传给下一层
- $C_t$ 会继续沿时间方向传递，作为长期记忆

所以单时间步可以理解成：

$(x_t,\ h_{t-1},\ C_{t-1}) \rightarrow (h_t,\ C_t)$

这就是 LSTM 单步前向传播的整体输入输出框架。

#### 3、单时间步前向传播的完整流程

##### 3.1 总体顺序先建立
在时间步 $t$，LSTM 一般按下面的顺序进行前向传播：

1. 计算遗忘门 $f_t$
2. 计算输入门 $i_t$
3. 计算候选记忆 $\tilde{C}_t$
4. 更新细胞状态 $C_t$
5. 计算输出门 $o_t$
6. 计算隐藏状态 $h_t$

这个顺序非常重要，后面会一直用到。📌


##### 3.2 第一步：计算遗忘门 $f_t$


遗忘门决定：

上一时刻细胞状态 $C_{t-1}$ 中，有多少内容应该保留。

公式是：

$ f_t = \sigma(W_f[h_{t-1}, x_t] + b_f) $

这里：

- $[h_{t-1}, x_t]$ 表示将两个向量拼接起来
- $W_f$ 是遗忘门的权重
- $b_f$ 是遗忘门的偏置
- $\sigma$ 是 Sigmoid 函数

因为 Sigmoid 输出范围在 $(0,1)$，所以 $f_t$ 就像一个“保留比例向量”：

- 接近 $1$：这一维旧记忆保留得多
- 接近 $0$：这一维旧记忆基本忘掉


##### 3.3 第二步：计算输入门 $i_t$

输入门决定：

当前时刻新产生的信息，有多少可以写入细胞状态。

公式是：

$ i_t = \sigma(W_i[h_{t-1}, x_t] + b_i) $

这里的 $i_t$ 也是一个 $0$ 到 $1$ 之间的门控向量。

它的作用不是生成新内容，而是决定“写入比例”。


##### 3.4 第三步：计算候选记忆 $\tilde{C}_t$

候选记忆表示：

当前时刻准备写入到细胞状态中的新内容是什么。

公式是：

$ \tilde{C}_t = \tanh(W_c[h_{t-1}, x_t] + b_c) $

这里用的是 tanh，因为它输出范围在 $(-1,1)$，适合表示有正有负的新记忆内容。

所以这一部分可以这样理解：

- $i_t$：决定“写多少”
- $\tilde{C}_t$：决定“写什么”


##### 3.5 第四步：更新细胞状态 $C_t$

这一步是 LSTM 最核心的一步。⭐

公式是：

$ C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t $

$\odot$ 表示逐元素相乘。

这个公式的含义非常清楚：

- $f_t \odot C_{t-1}$：保留下来的旧记忆
- $i_t \odot \tilde{C}_t$：写入进去的新记忆

所以新的细胞状态 $C_t$ 不是完全重新算出来的，而是：

旧记忆保留一部分 + 新记忆加入一部分

这也是 LSTM 比普通 RNN 更稳定的关键原因。


##### 3.6 第五步：计算输出门 $o_t$

输出门决定：

当前细胞状态中的哪些内容，可以作为当前时刻的隐藏状态输出。

公式是：

$ o_t = \sigma(W_o[h_{t-1}, x_t] + b_o) $

和前面的门一样，$o_t$ 也是一个 $0$ 到 $1$ 的门控向量。


##### 3.7 第六步：计算隐藏状态 $h_t$

当前隐藏状态由当前细胞状态和输出门共同决定：

$ h_t = o_t \odot \tanh(C_t) $

这个公式说明：

- 先对细胞状态 $C_t$ 做一次 tanh 压缩
- 再通过输出门 $o_t$ 控制哪些内容真正输出

所以：

- $C_t$：更像内部长期记忆
- $h_t$：更像当前时刻对外展示出来的短期状态


##### 3.8 总结
单时间步 LSTM 的前向传播可以总结成一句话：

先决定忘掉多少旧记忆，再决定加入多少新记忆，然后从更新后的记忆中挑选一部分作为当前输出。

#### 4、单时间步的维度计算

##### 4.1 先约定几个维度符号
为了后面不混乱，我们先约定：

- 输入特征维度：$d_x$
- 隐藏状态维度：$d_h$

那么在单时间步中：

- $x_t \in \mathbb{R}^{d_x}$
- $h_{t-1} \in \mathbb{R}^{d_h}$
- $C_{t-1} \in \mathbb{R}^{d_h}$


##### 4.2 拼接后的维度
因为各个门通常都基于 $[h_{t-1}, x_t]$ 来计算，所以拼接后向量维度是：

$ d_h + d_x $


##### 4.3 各个门和候选记忆的维度
为了让它们最终能够和 $C_{t-1}$ 做逐元素运算，所以：

- $f_t \in \mathbb{R}^{d_h}$
- $i_t \in \mathbb{R}^{d_h}$
- $\tilde{C}_t \in \mathbb{R}^{d_h}$
- $o_t \in \mathbb{R}^{d_h}$

这意味着各自的权重矩阵形状通常都是：

$ W_* \in \mathbb{R}^{d_h \times (d_h + d_x)} $

偏置则是：

$ b_* \in \mathbb{R}^{d_h} $


##### 4.4 输出状态维度
更新后：

- $C_t \in \mathbb{R}^{d_h}$
- $h_t \in \mathbb{R}^{d_h}$

所以你会发现：

LSTM 的所有门、候选记忆、细胞状态、隐藏状态，它们的维度都统一为隐藏维度 $d_h$。

这是因为它们都围绕“隐藏状态空间”在工作。

#### 5、从单时间步推广到多时间步

##### 5.1 多时间步的本质是什么
现在我们已经知道单时间步：

$(x_t,\ h_{t-1},\ C_{t-1}) \rightarrow (h_t,\ C_t)$

那么对于整个序列：

$x_1, x_2, x_3, \dots, x_T$

LSTM 所做的事情，其实就是在每个时间步重复同样的单步计算：

- 第 1 步：用 $x_1$、初始 $h_0$、初始 $C_0$ 算出 $h_1,\ C_1$
- 第 2 步：用 $x_2$、$h_1$、$C_1$ 算出 $h_2,\ C_2$
- 第 3 步：用 $x_3$、$h_2$、$C_2$ 算出 $h_3,\ C_3$
- ……
- 第 $T$ 步：用 $x_T$、$h_{T-1}$、$C_{T-1}$ 算出 $h_T,\ C_T$

也就是说：

同一个 LSTM 单元，在时间维度被重复使用。

这就是 LSTM 在时间维度上的展开。


##### 5.2 为什么说“整个序列只是重复单步”
因为 LSTM 在每个时间步内部执行的公式完全相同，只是输入不同：

- 当前输入变成了不同的 $x_t$
- 上一时刻状态变成了上一时刻算出来的 $h_{t-1}, C_{t-1}$

参数是不变的，变化的是时间步输入和状态。

也就是说：

- 同一个 $W_f$ 在所有时间步共享
- 同一个 $W_i$ 在所有时间步共享
- 同一个 $W_c$ 在所有时间步共享
- 同一个 $W_o$ 在所有时间步共享

这就是“循环网络”的核心特征之一：时间维参数共享。

#### 六、多时间步下的维度计算

##### 6.1 整个序列的输入
如果只看一条样本，一个长度为 $T$ 的序列输入可以写为：

$x_1, x_2, \dots, x_T$

其中每个：

$ x_t \in \mathbb{R}^{d_x} $

整个序列可以理解为：

$ X \in \mathbb{R}^{T \times d_x} $


##### 6.2 整个序列的隐藏状态输出
LSTM 在每个时间步都会产生一个隐藏状态：

$ h_1, h_2, \dots, h_T $

所以整个输出序列为：

$ H \in \mathbb{R}^{T \times d_h} $


##### 6.3 最后时刻状态
除了整个隐藏状态序列，LSTM 通常还会特别输出最后一个时间步的状态：

- 最终隐藏状态：$h_T$
- 最终细胞状态：$C_T$

这两个状态在很多任务中都很重要，比如：

- 文本分类：常常用最后时刻隐藏状态做分类
- Seq2Seq：常常把最后状态传给解码器

#### 7、加入 batch 之后怎么理解

##### 7.1 为什么要加入 batch
前面我们讨论的是“一次只输入一条序列”的情况。

但在实际训练中，我们通常不会一次只喂一条样本，而是会一次喂很多条样本，组成一个 batch，这样可以：

- 提高计算效率
- 更好利用 GPU 并行
- 让梯度更新更稳定

所以我们需要把前面的单条序列，扩展成“多条序列一起算”。


##### 7.2 加入 batch 后的输入张量
假设：

- batch size 为 $B$
- 序列长度为 $T$
- 输入维度为 $d_x$

那么如果采用 `batch_first=True` 的写法，输入张量通常是：

$ X \in \mathbb{R}^{B \times T \times d_x} $

含义是：

- 第 1 维：batch 中有多少条样本
- 第 2 维：每条样本有多少个时间步
- 第 3 维：每个时间步的特征维度

如果不用 `batch_first=True`，常见形式则是：

$ X \in \mathbb{R}^{T \times B \times d_x} $

这只是维度排列不同，本质是一样的。


##### 7.3 加入 batch 后单个时间步的输入
当我们固定某一个时间步 $t$ 时，这一时刻输入的不是一个向量了，而是 batch 中所有样本在第 $t$ 个时间步上的输入。

所以：

$ x_t \in \mathbb{R}^{B \times d_x} $

同时：

- $h_{t-1} \in \mathbb{R}^{B \times d_h}$
- $C_{t-1} \in \mathbb{R}^{B \times d_h}$

这意味着：

单时间步的逻辑没有变化，只是原来对一个向量做计算，现在变成了对一批向量并行计算。

#### 8、加入 batch 后单时间步的维度计算

##### 8.1 拼接后的维度
在 batch 情况下，拼接 $h_{t-1}$ 和 $x_t$ 后：

$ [h_{t-1}, x_t] \in \mathbb{R}^{B \times (d_h + d_x)} $


##### 8.2 各个门的输出维度
计算后得到：

- $f_t \in \mathbb{R}^{B \times d_h}$
- $i_t \in \mathbb{R}^{B \times d_h}$
- $\tilde{C}_t \in \mathbb{R}^{B \times d_h}$
- $o_t \in \mathbb{R}^{B \times d_h}$

更新后：

- $C_t \in \mathbb{R}^{B \times d_h}$
- $h_t \in \mathbb{R}^{B \times d_h}$

你会发现，和单样本情况相比，只是前面多了一个 batch 维。


##### 8.3 整个序列输出维度
对于整个 batch 的整个序列，如果使用 `batch_first=True`，那么输出隐藏状态序列通常是：

$ H \in \mathbb{R}^{B \times T \times d_h} $

最后时刻状态则通常是：

- $h_T \in \mathbb{R}^{B \times d_h}$
- $C_T \in \mathbb{R}^{B \times d_h}$

#### 9、把“单步、多步、batch”三者统一起来

##### 9.1 三个层次其实是一回事

第一层：单时间步

研究一个 LSTM 单元内部如何计算：

$(x_t,\ h_{t-1},\ C_{t-1}) \rightarrow (h_t,\ C_t)$

第二层：多时间步

把单时间步沿着时间方向重复 $T$ 次。

第三层：加入 batch

把“一个序列的计算”并行扩展为“多个序列同时计算”。


##### 9.2 最核心的理解
LSTM 的整个前向传播，本质上就是：

**单时间步计算 × 时间展开 × batch 并行**